# arg-position-back-functions — faded example 2: Complete back1 for out = x - k*y

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `arg-position-back-functions`. The last cell reports your progress on the `Backprop: Arg-position back funcs` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Arg-position back funcs` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`arg-position-back-functions`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "arg-position-back-functions"
DD_SUBTOPIC = "Backprop: Arg-position back funcs"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a forward mixes a constant into one position, the per-arg back fn for that position absorbs the constant. For `out = x - k*y` (with `k` a fixed scalar), `back0` is just `grad_out`, but `back1` must carry the `-k` factor. This shows the per-position back fn is shaped by the local derivative at *that* arg, not the other.

## Faded exercise 2

### Complete `scaled_sub_back1` for `out = x - k*y`

The constant `k = 3.0` is fixed. `scaled_sub_back0` (gradient w.r.t. `x`) is provided and simply returns `grad_out` since `d(x - k*y)/dx = 1`. Complete `scaled_sub_back1`, the gradient w.r.t. `y`: differentiate `x - k*y` w.r.t. `y` and apply the chain rule to `grad_out`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
K = 3.0


def scaled_sub_back0(grad_out, out, x, y):
    return grad_out


def scaled_sub_back1(grad_out, out, x, y):
    grad_y = -K * grad_out  # d(x - K*y)/dy = -K
    return grad_y


t.manual_seed(0)
x = t.randn(5, requires_grad=True)
y = t.randn(5, requires_grad=True)
out = x - K * y
grad_out = t.randn(5)
out.backward(grad_out)

gx = scaled_sub_back0(grad_out, out.detach(), x.detach(), y.detach())
gy = scaled_sub_back1(grad_out, out.detach(), x.detach(), y.detach())
print('grad_x match:', t.allclose(gx, x.grad))
print('grad_y match:', t.allclose(gy, y.grad))


def _test():
    K = 3.0
    t.manual_seed(0)
    x = t.randn(5, requires_grad=True)
    y = t.randn(5, requires_grad=True)
    out = x - K * y
    grad_out = t.randn(5)
    out.backward(grad_out)
    gy = scaled_sub_back1(grad_out, out.detach(), x.detach(), y.detach())
    assert gy.shape == y.shape
    assert t.allclose(gy, y.grad), 'scaled_sub_back1 must equal -K * grad_out'
    assert t.allclose(gy, -K * grad_out)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
K = 3.0


def scaled_sub_back0(grad_out, out, x, y):
    return grad_out


def scaled_sub_back1(grad_out, out, x, y):
    grad_y = -K * grad_out  # d(x - K*y)/dy = -K
    return grad_y


t.manual_seed(0)
x = t.randn(5, requires_grad=True)
y = t.randn(5, requires_grad=True)
out = x - K * y
grad_out = t.randn(5)
out.backward(grad_out)

gx = scaled_sub_back0(grad_out, out.detach(), x.detach(), y.detach())
gy = scaled_sub_back1(grad_out, out.detach(), x.detach(), y.detach())
print('grad_x match:', t.allclose(gx, x.grad))
print('grad_y match:', t.allclose(gy, y.grad))
```
</details>